# GOES vs floods: imagery + lightning vs verified flood events

**Research question (open):** do the clouds and deep convection moving over CONUS
line up with where floods are *confirmed* shortly after? This notebook puts four
things on one scroll-zoom map, for a **single date or a date range**:

1. **A GOES time-lapse** — every daytime frame (6/day, 16–21 UTC) in the window,
   reprojected to lat/lon and animated with a play/slider (bottom-left).
2. **GLM lightning** — the flashes within ±30 min of the displayed frame as
   yellow dots that advance with the animation (the label shows the ⚡ count).
3. **Verified floods** — NCEI storm-event hull polygons (human-confirmed
   occurrences) active `flood_lag_days` after the imagery window. Warnings and
   groundsource extents can be toggled in via `flood_sources`.
4. **A 25 × 25 km reference grid** over CONUS land — the sampling unit for
   later feature extraction.

All layers toggle from the **layer control** (top-right). This notebook is
**self-contained** — the first cell defines every helper. Requires the unified
parquet from [`explore/flood_data_explore.ipynb`](explore/flood_data_explore.ipynb)
and the GLM day-parquets from `floodlens.download.glm`.

## Setup & helpers — run this cell once

In [ ]:
# ---------------------------------------------------------------------------
# Self-contained helpers — run this cell once.
# GOES geostationary imagery -> reprojected web-map overlay, a 25 km CONUS land
# grid, the unified flood layer (storm events / warnings / groundsource), GLM
# lightning flashes, and a synced play/slider time-lapse, all on one map.
# ---------------------------------------------------------------------------
import base64
import io
import json
import urllib.request
from datetime import date, datetime, timedelta
from pathlib import Path

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyproj
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from branca.element import MacroElement
from folium.raster_layers import ImageOverlay
from jinja2 import Template
from PIL import Image
from shapely.geometry import box

# ---- paths (work whether cwd is the repo root or notebooks/) ----
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = Path("/mnt/disk1/goes-data")             # GOES NetCDFs
GLM_DIR = Path("/mnt/disk1/glm-data")               # GLM per-day flash parquets
AUX_DIR = DATA_DIR / "aux"                          # cached boundaries / grids
# Unified flood layer (built in explore/flood_data_explore.ipynb)
UNIFIED_PARQUET = ROOT / "data/flood_warnings/floods_unified.parquet"

# CONUS land box (lon_min, lon_max, lat_min, lat_max) — drops AK/HI/PR/territories
CONUS_BBOX = (-125.0, -66.5, 24.0, 50.0)
CONUS_ALBERS = 5070                                 # equal-area metres for the grid

US_STATES_GEOJSON = AUX_DIR / "us-states.geojson"
US_STATES_URL = (
    "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/"
    "master/data/geojson/us-states.json"
)
NON_CONUS_STATES = {"Alaska", "Hawaii", "Puerto Rico"}

# Per-source draw style for the unified flood layer
SOURCE_STYLE = {
    "storm_event":  {"label": "storm events (verified)", "color": "#6a3d9a",
                     "fillColor": "#6a3d9a", "fillOpacity": 0.45, "weight": 1.0},
    "ff_warning":   {"label": "flash-flood warnings", "color": "#e31a1c",
                     "fillColor": "#e31a1c", "fillOpacity": 0.15, "weight": 1.0},
    "fa_warning":   {"label": "areal-flood warnings", "color": "#ff7f00",
                     "fillColor": "#ff7f00", "fillOpacity": 0.15, "weight": 1.0},
    "groundsource": {"label": "groundsource (news reports)", "color": "#0b4dd6",
                     "fillColor": "#1f78ff", "fillOpacity": 0.55, "weight": 0.5},
}

FLASH_HALF_WINDOW = pd.Timedelta(minutes=30)    # flashes shown around each frame
MAX_DOTS_PER_FRAME = 5000                       # sampled above this, for the HTML

_BLANK = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAA"
          "C0lEQVR42mNk+M8AAAMBAQDJ/pLvAAAAAElFTkSuQmCC")  # 1x1 transparent


# ---- file discovery ----
def _scan_token(p):
    """The filename's s{YYYYDDDHHMMSSf} scan-start token."""
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def _stamp(p):
    """'YYYY-MM-DD HH:MM UTC' from a filename's scan-start token."""
    t = _scan_token(p)
    if t.startswith("s") and len(t) >= 12:
        d = datetime.strptime(t[1:8], "%Y%j")
        return f"{d:%Y-%m-%d} {t[8:10]}:{t[10:12]} UTC"
    return p.name


def _frame_time(p):
    """Scan-start datetime from a GOES filename's s{YYYYDDDHHMM...} token."""
    return datetime.strptime(_scan_token(p)[1:12], "%Y%j%H%M")


def find_files(dt, data_dir=DATA_DIR):
    """All GOES NetCDFs for a date, sorted by scan time (both satellites)."""
    pat = f"*/{dt.year}/{dt.month:02d}/{dt.day:02d}/*.nc"
    return sorted(data_dir.glob(pat), key=_scan_token)


# ---- band access & scaling ----
def cmi(ds, n):
    """The CMI_C<n> band (reflectance or brightness temperature)."""
    return ds[f"CMI_C{n:02d}"]


def _cmap(da):
    bt = str(da.attrs.get("units", "")).strip().upper().startswith("K")
    return "gray_r" if bt else "gray"               # IR: cold cloud tops = white


def _stretch(a, vmin=None, vmax=None):
    lo = np.nanpercentile(a, 2) if vmin is None else vmin
    hi = np.nanpercentile(a, 98) if vmax is None else vmax
    return float(lo), float(hi)


def _norm(a, lo, hi):
    return np.clip((a - lo) / (hi - lo), 0, 1) if hi > lo else np.zeros_like(a)


# ---- crop + reproject geostationary -> lat/lon, then encode an RGBA overlay ----
def _geos(ds):
    return pyproj.CRS.from_cf(dict(ds["goes_imager_projection"].attrs))


def crop_lonlat(ds, lon_min, lon_max, lat_min, lat_max):
    """Subset to a lon/lat box (degrees) before reprojecting — faster + zoomed."""
    h = ds["goes_imager_projection"].attrs["perspective_point_height"]
    tf = pyproj.Transformer.from_crs("EPSG:4326", _geos(ds), always_xy=True)
    xs, ys = tf.transform([lon_min, lon_max, lon_min, lon_max],
                          [lat_min, lat_min, lat_max, lat_max])
    xs, ys = np.array(xs) / h, np.array(ys) / h
    xs, ys = xs[np.isfinite(xs)], ys[np.isfinite(ys)]
    if xs.size == 0 or ys.size == 0:
        raise ValueError("box is off the Earth disk for this satellite")
    return ds.sel(x=slice(xs.min(), xs.max()), y=slice(ys.max(), ys.min()))


def reproject_bands(ds, bands):
    """Reproject bands to EPSG:4326; return (DataArray[band], [[s, w], [n, e]])."""
    h = ds["goes_imager_projection"].attrs["perspective_point_height"]
    da = xr.concat([cmi(ds, n).reset_coords(drop=True) for n in bands], dim="band")
    da = da.assign_coords(x=ds["x"] * h, y=ds["y"] * h)
    da = da.rio.write_crs(_geos(ds)).rio.set_spatial_dims(x_dim="x", y_dim="y")
    da = da.rio.reproject("EPSG:4326", nodata=np.nan)
    da = da.sortby("y", ascending=False).sortby("x")
    minx, miny, maxx, maxy = da.rio.bounds()
    return da, [[miny, minx], [maxy, maxx]]


def _rgba_single(a, cmap, lo, hi):
    finite = np.isfinite(a)
    sm = plt.cm.ScalarMappable(norm=plt.Normalize(lo, hi), cmap=cmap)
    rgba = sm.to_rgba(np.nan_to_num(a, nan=lo), bytes=True)
    rgba[..., 3] = np.where(finite, 255, 0).astype("uint8")
    return rgba


def _rgba_rgb(arr3, gamma=2.2):
    chans, alpha = [], np.ones(arr3.shape[1:], dtype=bool)
    for a in arr3:
        chans.append(_norm(a, *_stretch(a)))
        alpha = alpha & np.isfinite(a)
    rgb = np.nan_to_num(np.clip(np.dstack(chans) ** (1 / gamma), 0, 1))
    a8 = np.where(alpha, 255, 0).astype("uint8")
    return np.dstack([(rgb * 255).astype("uint8"), a8])


def _data_uri(rgba):
    buf = io.BytesIO()
    Image.fromarray(rgba, mode="RGBA").save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def _downsample(rgba, max_px):
    """Shrink an RGBA frame so the embedded HTML stays small."""
    h, w = rgba.shape[:2]
    if max(h, w) <= max_px:
        return rgba
    s = max_px / max(h, w)
    im = Image.fromarray(rgba, "RGBA").resize(
        (max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    return np.asarray(im)


def _frame_rgba(ds, band, rgb, crop):
    """Reproject one dataset to an RGBA array + lat/lon bounds for an overlay."""
    sub = crop_lonlat(ds, *crop) if crop else ds
    if rgb is not None:
        da_ll, bounds = reproject_bands(sub, list(rgb))
        return _rgba_rgb(da_ll.values), bounds
    da_ll, bounds = reproject_bands(sub, [band])
    a = da_ll.isel(band=0).values
    return _rgba_single(a, _cmap(cmi(sub, band)), *_stretch(a)), bounds


# ---- 25 km CONUS land grid ----
def conus_land(states_geojson=US_STATES_GEOJSON):
    """CONUS land polygon (lower-48 + DC) in EPSG:4326 (downloads once if missing)."""
    if not states_geojson.exists():
        states_geojson.parent.mkdir(parents=True, exist_ok=True)
        print(f"downloading US states boundary -> {states_geojson}")
        urllib.request.urlretrieve(US_STATES_URL, states_geojson)
    g = gpd.read_file(states_geojson)
    return g[~g["name"].isin(NON_CONUS_STATES)].to_crs(4326)


def conus_grid(cell_km=25.0, cache=True):
    """Fishnet of cell_km square cells over CONUS land (EPSG:4326, cached).

    Cells are built in CONUS Albers (EPSG:5070) so they are genuinely cell_km on
    a side, then kept where they intersect land. Each row has a cell_id — the
    natural sampling unit for later feature extraction.
    """
    cache_path = AUX_DIR / f"conus_grid_{int(cell_km)}km.parquet"
    if cache and cache_path.exists():
        return gpd.read_parquet(cache_path)
    land = conus_land().to_crs(CONUS_ALBERS)
    land_union = land.union_all()
    step = cell_km * 1000.0
    minx, miny, maxx, maxy = land.total_bounds
    cells, ids = [], []
    for r, y0 in enumerate(np.arange(np.floor(miny / step) * step, maxy + step, step)):
        for c, x0 in enumerate(np.arange(np.floor(minx / step) * step, maxx + step, step)):
            cells.append(box(x0, y0, x0 + step, y0 + step))
            ids.append(f"r{r:03d}c{c:03d}")
    grid = gpd.GeoDataFrame({"cell_id": ids}, geometry=cells, crs=CONUS_ALBERS)
    grid = grid[grid.intersects(land_union)].reset_index(drop=True).to_crs(4326)
    if cache:
        AUX_DIR.mkdir(parents=True, exist_ok=True)
        grid.to_parquet(cache_path)
    return grid


# ---- unified flood layer (storm events + warnings + groundsource) ----
def load_unified(start, end=None, bbox=CONUS_BBOX, sources=None,
                 parquet=UNIFIED_PARQUET):
    """Unified flood rows active in [start, end] (inclusive days) within bbox.

    Built in explore/flood_data_explore.ipynb (kind, source, event_id,
    phenomena, issue_date, expire_date, area_km2, flood_cause, geometry).
    "Active" = [issue_date, expire_date] overlaps the window. Pass `sources`
    to keep some of storm_event / ff_warning / fa_warning / groundsource.
    """
    if not parquet.exists():
        raise FileNotFoundError(
            f"{parquet} not found — run the 'Persist the unified frame' cell in "
            "explore/flood_data_explore.ipynb first.")
    t0 = pd.Timestamp(start)
    t1 = (pd.Timestamp(end) if end is not None else t0) + pd.Timedelta(days=1)
    g = gpd.read_parquet(parquet)
    g = g[(g["issue_date"] < t1) & (g["expire_date"] >= t0)]
    if sources is not None:
        g = g[g["source"].isin(sources)]
    lon_min, lon_max, lat_min, lat_max = bbox
    return g.cx[lon_min:lon_max, lat_min:lat_max].reset_index(drop=True)


# ---- GLM lightning flashes (per-day parquets from floodlens.download.glm) ----
def _glm_path(dt, glm_dir=GLM_DIR):
    return glm_dir / str(dt.year) / f"glm_flashes_{dt:%Y%m%d}.parquet"


def load_glm(dt, bbox=CONUS_BBOX, glm_dir=GLM_DIR):
    """One day of GLM flashes clipped to bbox (times, lat/lon, energy, area)."""
    df = pd.read_parquet(_glm_path(dt, glm_dir))
    lon_min, lon_max, lat_min, lat_max = bbox
    return df[df["lon"].between(lon_min, lon_max)
              & df["lat"].between(lat_min, lat_max)].reset_index(drop=True)


# ---- folium layers ----
def grid_layer(grid, name="25 km grid"):
    """Thin, unfilled grid outlines as a toggleable GeoJson layer."""
    return folium.GeoJson(
        grid[["cell_id", "geometry"]].to_json(), name=name,
        style_function=lambda _f: {"color": "#444", "weight": 0.4,
                                   "fill": False, "opacity": 0.5},
        tooltip=folium.GeoJsonTooltip(fields=["cell_id"], aliases=["cell"]))


def unified_layers(floods):
    """One toggleable GeoJson layer per flood source."""
    layers = []
    for src, style in SOURCE_STYLE.items():
        sub = floods[floods["source"] == src]
        if not len(sub):
            continue
        gj = sub[["event_id", "phenomena", "issue_date", "expire_date",
                  "area_km2", "flood_cause", "geometry"]].copy()
        gj["issue_date"] = gj["issue_date"].astype(str)
        gj["expire_date"] = gj["expire_date"].astype(str)
        gj["flood_cause"] = gj["flood_cause"].fillna("")
        gj["area_km2"] = gj["area_km2"].round(1)
        sf = {k: style[k] for k in ("color", "fillColor", "fillOpacity", "weight")}
        layers.append(folium.GeoJson(
            gj.to_json(), name=f"{style['label']} ({len(sub):,})",
            style_function=lambda _f, sf=sf: sf,
            tooltip=folium.GeoJsonTooltip(
                fields=["phenomena", "issue_date", "expire_date", "area_km2",
                        "flood_cause"],
                aliases=["type", "from", "to", "km2", "cause"])))
    return layers


def _base_map(bounds):
    """Scroll-zoom folium map with OSM / light / satellite basemaps."""
    (s, w), (n, e) = bounds
    m = folium.Map(location=[(s + n) / 2, (w + e) / 2], zoom_start=5,
                   tiles="OpenStreetMap")
    folium.TileLayer("CartoDB positron", name="light").add_to(m)
    folium.TileLayer(
        tiles=("https://server.arcgisonline.com/ArcGIS/rest/services/"
               "World_Imagery/MapServer/tile/{z}/{y}/{x}"),
        attr="Esri World Imagery", name="satellite").add_to(m)
    return m


# ---- synced time-lapse: cloud frames + that frame's GLM flash dots ----
class _FlashCloudLapse(MacroElement):
    """Play/slider that cycles cloud frames AND that frame's flash dots."""

    _template = Template("""
        {% macro script(this, kwargs) %}
        (function(){
          var fr={{this.frames_json}}, lb={{this.labels_json}}, fl={{this.flashes_json}};
          var ov={{this.overlay_name}}, gp={{this.group_name}};
          var mp={{this._parent.get_name()}};
          var nm="{{this.get_name()}}", iv={{this.interval}}, i=0, t=null, pl=false;
          var cv=L.canvas({padding:0.2});
          var c=L.control({position:'bottomleft'});
          c.onAdd=function(){
            var d=L.DomUtil.create('div','');
            d.style.cssText='background:rgba(255,255,255,.88);padding:6px 8px;'
              +'font:12px sans-serif;border-radius:4px';
            d.innerHTML='<button id="b_'+nm+'">&#9658;</button> '
              +'<input id="s_'+nm+'" type="range" min="0" max="'+(fr.length-1)
              +'" value="0" style="width:240px;vertical-align:middle"> '
              +'<span id="l_'+nm+'"></span>';
            L.DomEvent.disableClickPropagation(d);
            return d;
          };
          c.addTo(mp);
          function dots(k){if(!gp)return;gp.clearLayers();
            fl[k].forEach(function(p){
              L.circleMarker([p[0],p[1]],{renderer:cv,radius:2.2,weight:.6,
                color:'#8a5a00',fillColor:'#ffd400',fillOpacity:.85}).addTo(gp);});}
          function show(k){i=(k+fr.length)%fr.length;ov.setUrl(fr[i]);dots(i);
            document.getElementById('s_'+nm).value=i;
            document.getElementById('l_'+nm).innerText=
              lb[i]+(gp?' \\u00b7 \\u26a1 '+fl[i].length:'');}
          function stop(){pl=false;document.getElementById('b_'+nm).innerHTML='&#9658;';
            clearInterval(t);}
          function go(){pl=true;
            document.getElementById('b_'+nm).innerHTML='&#10074;&#10074;';
            t=setInterval(function(){show(i+1);},iv);}
          setTimeout(function(){
            show(0);
            document.getElementById('b_'+nm).onclick=function(){pl?stop():go();};
            document.getElementById('s_'+nm).oninput=function(e){stop();
              show(parseInt(e.target.value));};
            go();
          },300);
        })();
        {% endmacro %}
    """)

    def __init__(self, overlay_name, group_name, frames, labels, flash_pts,
                 interval=800):
        super().__init__()
        self._name = "FlashCloudLapse"
        self.overlay_name = overlay_name
        self.group_name = group_name or "null"
        self.frames_json = json.dumps(frames)
        self.labels_json = json.dumps(labels)
        self.flashes_json = json.dumps(flash_pts)
        self.interval = interval


# ---- the combined view ----
def goes_vs_floods_timelapse(start, end=None, band=13, rgb=None, crop=None,
                             show_grid=True, grid_cell_km=25.0,
                             flood_sources=("storm_event",), flood_lag_days=1,
                             show_lightning=True, max_frames=48, max_px=1400,
                             interval=800, opacity=0.8,
                             max_dots=MAX_DOTS_PER_FRAME, data_dir=DATA_DIR):
    """GOES time-lapse + GLM lightning + flood layer for a date or date range.

    Time-lapses every GOES frame from `start` to `end` (inclusive; 6 daytime
    frames/day), with the GLM flashes within +/-30 min of each frame as yellow
    dots that advance with the play/slider. The unified flood layer active in
    [start + flood_lag_days, end + flood_lag_days] is drawn per source —
    default just the **verified storm events**; pass e.g.
    flood_sources=("storm_event", "ff_warning", "groundsource") for more, or
    None for all. band=N (13 = cold cloud tops, 8 = water vapour) or
    rgb=(2, 3, 1) for true colour. crop=(lon_min, lon_max, lat_min, lat_max).
    """
    end = end or start
    day_list = [start + timedelta(days=i) for i in range((end - start).days + 1)]
    files = [p for d in day_list for p in find_files(d, data_dir)]
    if not files:
        raise FileNotFoundError(f"No GOES files for {start}..{end} under {data_dir}")
    if len(files) > max_frames:
        print(f"[note] {len(files)} frames -> keeping the first {max_frames} "
              f"(raise max_frames to keep more; HTML grows ~0.5 MB/frame)")
        files = files[:max_frames]
    view_bbox = crop if crop else CONUS_BBOX

    flashes = None
    if show_lightning:
        have = [d for d in day_list if _glm_path(d).exists()]
        if have:
            flashes = pd.concat([load_glm(d, bbox=view_bbox) for d in have],
                                ignore_index=True)
        if len(have) < len(day_list):
            print(f"[note] GLM parquets missing for "
                  f"{len(day_list) - len(have)} day(s) — "
                  "run floodlens.download.glm build")

    n_fl = f", {len(flashes):,} GLM flashes in view" if flashes is not None else ""
    print(f"{len(files)} cloud frame(s) {day_list[0]} -> {day_list[-1]}{n_fl}; "
          "reprojecting...")

    frames, labels, flash_pts, bounds = [], [], [], None
    for j, p in enumerate(files):
        ds = xr.open_dataset(p, decode_times=False)
        try:
            rgba, b = _frame_rgba(ds, band, rgb, crop)
        finally:
            ds.close()
        bounds = bounds or b
        frames.append(_data_uri(_downsample(rgba, max_px)))
        labels.append(_stamp(p))
        if flashes is not None:
            t = _frame_time(p)
            win = flashes[(flashes["time_start"] >= t - FLASH_HALF_WINDOW)
                          & (flashes["time_start"] < t + FLASH_HALF_WINDOW)]
            if len(win) > max_dots:
                win = win.sample(max_dots, random_state=0)
            flash_pts.append(
                win[["lat", "lon"]].astype(float).round(3).values.tolist())
        else:
            flash_pts.append([])
        print(f"  {j + 1}/{len(files)}  {labels[-1]}", end="\r")
    print()

    lag = timedelta(days=flood_lag_days)
    floods = load_unified(day_list[0] + lag, day_list[-1] + lag,
                          bbox=view_bbox,
                          sources=list(flood_sources) if flood_sources else None)
    print(f"{len(floods)} flood row(s) active {day_list[0] + lag} -> "
          f"{day_list[-1] + lag} by source: "
          f"{floods.source.value_counts().to_dict()}")

    m = _base_map(bounds)
    ov = ImageOverlay(_BLANK, bounds=bounds, opacity=opacity,
                      name=f"GOES {day_list[0]}..{day_list[-1]}")
    ov.add_to(m)
    if show_grid:
        lon_min, lon_max, lat_min, lat_max = view_bbox
        grid = conus_grid(grid_cell_km).cx[lon_min:lon_max, lat_min:lat_max]
        grid_layer(grid, f"{int(grid_cell_km)} km grid").add_to(m)
    for layer in unified_layers(floods):
        layer.add_to(m)
    fg_name = None
    if flashes is not None:
        n_shown = sum(len(pts) for pts in flash_pts)
        fg = folium.FeatureGroup(
            name=f"GLM flashes ±30 min of frame ({n_shown:,} dots)")
        fg.add_to(m)
        fg_name = fg.get_name()
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds(bounds)
    m.add_child(_FlashCloudLapse(ov.get_name(), fg_name, frames, labels,
                                 flash_pts, interval))
    return m

In [ ]:
# Use the project's config grid (model-consistent) for the grid overlays below.
# Redefine conus_grid() to pull from config.build_grid_cells (currently 50 km)
# so the map's grid matches notebooks/model -- one grid definition project-wide.
from floodlens.config import build_grid_cells


def conus_grid(cell_km=None, cache=False):
    "CONUS grid from config.build_grid_cells (model CELL_KM); args kept for compat."
    return build_grid_cells()[0]


## Single day

Default view: **Hurricane Ida** making landfall (2021-08-31), with the storm
events confirmed the next day. Band 13 (clean IR) makes cold storm-cloud tops
bright; watch the flash dots ride under them.

In [ ]:
m = goes_vs_floods_timelapse(
    date(2021, 8, 31),          # Hurricane Ida
    band=13,                    # clean IR: cold cloud tops = bright
    flood_lag_days=1,           # verified floods active the *next* day
)
m

## A range of dates

Pass `end=` for a multi-day time-lapse — e.g. **Hurricane Helene** crossing the
Southeast (2024-09-26 → 09-27, 12 frames). Cropped so the embedded map stays
light; set `crop=None` for full CONUS (bigger HTML).

In [ ]:
goes_vs_floods_timelapse(
    date(2024, 9, 26), end=date(2024, 9, 27),    # Hurricane Helene
    band=13,
    crop=(-92, -75, 24, 38),                     # Gulf -> Appalachians
    flood_lag_days=1,
)

## All flood layers at once

`flood_sources=None` draws every source — verified storm events (purple),
FF/FA warnings (red/orange), groundsource news-report extents (blue) — each as
its own toggleable layer. Useful for eyeballing how the label sources disagree.

In [ ]:
goes_vs_floods_timelapse(
    date(2018, 1, 1),
    band=1,
    flood_sources=None,         # storm events + warnings + groundsource
)

## The pieces, on their own

- `conus_grid(25)` → cached `GeoDataFrame` of 25 km land cells (`cell_id`, geometry).
- `load_unified(start, end)` → unified flood rows active in the window
  (filter with `sources=`).
- `load_glm(date)` → one day of GLM flashes.
- `find_files(date)` → the GOES NetCDFs for a day.

In [ ]:
grid = conus_grid(25)
floods = load_unified(date(2021, 9, 1), sources=["storm_event"])
fl = load_glm(date(2021, 8, 31))
print(f"grid: {len(grid):,} cells of 25 km")
print(f"verified storm events active 2021-09-01: {len(floods):,}")
print(f"GLM flashes on 2021-08-31 (CONUS): {len(fl):,}")
floods.head(3)

## Notes & next steps

- **Grid is the sampling unit.** Each 25 km `cell_id` is a candidate location for
  per-cell features (cloud-top temperature, water vapour, flash density) vs. a
  next-day flood label. Cached at `/mnt/disk1/goes-data/aux/conus_grid_25km.parquet`.
- **`flood_sources` picks the label layer**: `("storm_event",)` (default,
  verified occurrences), `("ff_warning", "fa_warning")` (predictions),
  `("groundsource",)` (news-report extents), or `None` for all — the sources
  agree only partially, so the choice matters.
- **`flood_lag_days`** shifts the flood window relative to the imagery (try 0
  for same-day, 2 for slower river flooding).
- **Daytime only.** Frames are 16–21 UTC; IR (band 13) and water vapour (band 8)
  work regardless, true colour (`rgb=(2, 3, 1)`) only shows the lit portion.
- **Frame budget:** each frame adds ~0.5 MB to the notebook; ranges longer than
  `max_frames=48` (8 days) are truncated with a note.